In [0]:
import re
import uuid
import traceback
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType, LongType, TimestampType)

### Configurations

In [0]:
LOG_TABLE    = "iics.casemix.json_large_files_load_log"
FILE_EXT     = (".json",)
RECURSIVE    = True        # scan sub-folders of src_path
SRC_SYSTEM   = "NPHC"      # value for the src_system column

RELOAD_ALL = "false"

RUN_ID      = str(uuid.uuid4())
EXECUTED_BY = spark.sql("SELECT current_user()").first()[0]
print(f"run_id={RUN_ID}  user={EXECUTED_BY}  reload_all={RELOAD_ALL}")

In [0]:
%sql
select * from iics.casemix.json_large_files_config_test

### Temp Remove in prod

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {LOG_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS iics.casemix.json_large_file_test")

### Log Configuration

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    run_id            STRING,
    src               STRING,
    src_path          STRING,
    file_name         STRING,
    file_path         STRING,
    file_size_mb      DOUBLE,
    file_modified_ms  BIGINT,
    target_table      STRING,
    status            STRING,     -- SUCCESS | FAILED | SKIPPED
    error_type        STRING,
    error_message     STRING,
    started_at        TIMESTAMP,
    ended_at          TIMESTAMP,
    executed_by       STRING
) USING DELTA
""")

LOG_SCHEMA = StructType([
    StructField("run_id", StringType()),
    StructField("src", StringType()),
    StructField("src_path", StringType()),
    StructField("file_name", StringType()),
    StructField("file_path", StringType()),
    StructField("file_size_mb", DoubleType()),
    StructField("file_modified_ms", LongType()),
    StructField("target_table", StringType()),
    StructField("status", StringType()),
    StructField("error_type", StringType()),
    StructField("error_message", StringType()),
    StructField("started_at", TimestampType()),
    StructField("ended_at", TimestampType()),
    StructField("executed_by", StringType()),
])

def log_event(cfg, f, target, status, started, error=None):
    """Append one row to the log table. Logging must never break the load."""
    try:
        row = (
            RUN_ID, cfg["src"], cfg["src_path"],
            f.name if f else None, f.path if f else None,
            f.size / (1024.0 * 1024.0) if f else None,
            int(f.modificationTime) if f else None,
            target, status,
            type(error).__name__ if error else None,
            (str(error)[:4000] + "\n" + traceback.format_exc()[-2000:]) if error else None,
            started, datetime.now(), EXECUTED_BY,
        )
        spark.createDataFrame([row], LOG_SCHEMA).write.mode("append").saveAsTable(LOG_TABLE)
    except Exception as log_err:
        print(f"WARNING: could not write log row: {log_err}")

### Target table & file name

In [0]:
def create_target_table(target):
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target} (
        src_system      STRING,
        file_id         BIGINT,      -- serial no.
        file_name       STRING,
        file_date       TIMESTAMP,   -- uploaded in source
        file_path       STRING,
        dt_datafile     DATE,        -- middle date from file name
        file_size_mb    DOUBLE,
        file_structure  VARIANT,
        table_name      STRING,      -- from file name
        created_by      STRING,
        created_on      TIMESTAMP
    ) USING DELTA
    """)

# <table_name>_<start yyyymmdd>_<MID yyyymmdd>_<end yyyymmdd>_<hhmmss>.json
FILE_NAME_PATTERN = re.compile(
    r"^(?P<table_name>.+?)_\d{8}_(?P<dt_datafile>\d{8})_\d{8}_\d{6}\.json$", re.IGNORECASE)

def parse_file_name(file_name):
    """'case_data_006_20220605_20220620_20260621_235954.json' -> ('case_data_006', date(2022, 6, 20))"""
    m = FILE_NAME_PATTERN.match(file_name)
    if not m:
        raise ValueError(f"Bad file name '{file_name}'. "
                         "Expected <table>_<yyyymmdd>_<yyyymmdd>_<yyyymmdd>_<hhmmss>.json")
    return m["table_name"], datetime.strptime(m["dt_datafile"], "%Y%m%d").date()

In [0]:
def list_files(path, recursive=True):
    """Return all files under path (optionally recursing into sub-folders)."""
    out = []
    for item in dbutils.fs.ls(path):
        if item.isDir():
            if recursive:
                out.extend(list_files(item.path, recursive))
        elif item.name.lower().endswith(FILE_EXT):
            out.append(item)
    return sorted(out, key=lambda x: x.path)

def already_loaded(target):
    """Set of (file_path, file_modified_ms) already loaded successfully into target."""
    rows = (spark.table(LOG_TABLE)
            .filter((F.col("status") == "SUCCESS") & (F.col("target_table") == target))
            .select("file_path", "file_modified_ms").distinct().collect())
    return {(r.file_path, r.file_modified_ms) for r in rows}

def load_file(f, target):
    """Read one JSON file, capture its structure, append one metadata row to target."""
    table_name, dt_datafile = parse_file_name(f.name)      # fails fast on a bad name

    file_structure = spark.read.option("multiLine", "true").json(f.path).schema.json()

    if RELOAD_ALL:                                          # replace the old row instead of duplicating it
        spark.sql(f"DELETE FROM {target} WHERE file_path = :p", args={"p": f.path})
    file_id = spark.sql(f"SELECT COALESCE(MAX(file_id), 0) + 1 FROM {target}").first()[0]

    (spark.createDataFrame([Row(
            src_system     = SRC_SYSTEM,
            file_id        = file_id,
            file_name      = f.name,
            file_date      = datetime.fromtimestamp(f.modificationTime / 1000),
            file_path      = f.path,
            dt_datafile    = dt_datafile,
            file_size_mb   = f.size / (1024.0 * 1024.0),
            file_structure = file_structure,
            table_name     = table_name,
            created_by     = EXECUTED_BY,
        )])
        .withColumn("file_structure", F.parse_json("file_structure"))
        .withColumn("created_on", F.current_timestamp())
        .select("src_system", "file_id", "file_name", "file_date", "file_path", "dt_datafile",
                "file_size_mb", "file_structure", "table_name", "created_by", "created_on")
        .write.mode("append").saveAsTable(target))

### Main

In [0]:
configs = [r.asDict() for r in
           spark.table(CONFIG_TABLE).filter("is_active = 1").collect()]
print(f"{len(configs)} active config row(s)")

summary = {"SUCCESS": 0, "FAILED": 0, "SKIPPED": 0}

for cfg in configs:
    target = f"{cfg['tgt_catalg']}.{cfg['tgt_schema']}.{cfg['tgt_table']}"
    print(f"\n=== {cfg['src_path']} -> {target} ===")

    start_time = datetime.now()
    try:
        create_target_table(target)
        files = list_files(cfg["src_path"], RECURSIVE)
        done = set() if RELOAD_ALL else already_loaded(target)
    except Exception as e:
        log_event(cfg, None, target, "FAILED", start_time, e)
        summary["FAILED"] += 1
        print(f"FAILED preparing {cfg['src_path']}: {e}")
        continue

    print(f"{len(files)} file(s) found")
    for f in files:
        started = datetime.now()
        if (f.path, int(f.modificationTime)) in done:
            log_event(cfg, f, target, "SKIPPED", started)
            summary["SKIPPED"] += 1
            print(f"SKIPPED  {f.name} (already loaded)")
            continue
        try:
            load_file(f, target)
            log_event(cfg, f, target, "SUCCESS", started)
            summary["SUCCESS"] += 1
            print(f"SUCCESS  {f.name}")
        except Exception as e:
            log_event(cfg, f, target, "FAILED", started, e)
            summary["FAILED"] += 1
            print(f"FAILED   {f.name}: {str(e)[:300]}")

print("\nSummary:", summary)

In [0]:
# ---------- Review this run's log ----------
display(spark.table(LOG_TABLE)
        .filter(F.col("run_id") == RUN_ID)
        .orderBy("started_at"))

# Fail the notebook/job if any file failed (details are in the log table)
if summary["FAILED"] > 0:
    raise Exception(f"{summary['FAILED']} file(s) failed. See {LOG_TABLE} where run_id = '{RUN_ID}'")

In [0]:
%sql
SELECT * 
FROM iics.casemix.json_large_file_test
ORDER BY file_id